In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel,Field
from typing import Literal

# Libs for Agents

from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain.agents import create_agent
from langchain.tools import tool


C:\Users\alok9\AppData\Local\Temp\ipykernel_16668\3981955816.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import GoogleSerperAPIWrapper


In [3]:
class FlowState(BaseModel):
    question:str= Field(description="User Ask a Question ?")
    category: Literal["coding", "weather", "Google_search"] = Field(default="Google_search")
    ans : str = Field(default="")
    

In [4]:
class question_category(BaseModel):
    category:Literal["coding","weather", "Google_search"] = Field(default="Google_search", 
                                                                  description="question category is conditional")
    

In [5]:
llm = ChatGroq(model="openai/gpt-oss-20b")

In [6]:
### Define Your Agents - GoogleAgent, Weather Agent

search = GoogleSerperAPIWrapper(SERPER_API_KEY="de81c8dfc0e6698d954b0bf4b70343a83a293c45")
tools = [search.run]

google_agent = create_agent(
    model=llm,
    tools = tools,
    system_prompt="You are a agent and can search for any question on google."
)


## Weather Agent

@tool
def get_weather(city:str):
    """It provide real time weather details for any city"""
    
    return f"The current temperature in {city} is  23.c"



weather_agent = create_agent(
    model=llm,
    tools = tools,
    system_prompt="You are a agent and provide real time weather details."
)





In [7]:
def  check_question_category(state:FlowState):
    st_llm = llm.with_structured_output(question_category)
    res = st_llm.invoke(f" I want to know my question category , question is :{state.question}.if you are not sure than say Google search ")
    state.category = res.category
    
    return state

In [8]:
flow = FlowState(question= "what is the temperature of india  ?" )
check_question_category(flow)

FlowState(question='what is the temperature of india  ?', category='weather', ans='')

In [9]:
def routes(state: FlowState) -> Literal["coding", "weather", "Google_search"]:
    return state.category

In [10]:
def coding_node(state:FlowState) -> FlowState:
    res = llm.invoke(f" You are a coding expert :{state.question}")
    state.ans = res.content
    return state

In [23]:
def weather_node(state:FlowState)-> FlowState:
    res = weather_agent.invoke({"messages":[
        {"role":"user", "content": state.question}
    ]})
    state.ans = res["messages"][-1].content
    return state

In [24]:
def Google_search_node(state:FlowState)-> FlowState:
    res = weather_agent.invoke({"messages":[
            {"role":"user", "content": state.question}
        ]})
    state.ans = res["messages"][-1].content
    return state

In [25]:
graph= StateGraph(FlowState)

In [26]:
graph.add_node("check_question_category", check_question_category)
graph.add_node("coding", coding_node )
graph.add_node("weather", weather_node)
graph.add_node("Google_search", Google_search_node)


graph.add_edge(START, "check_question_category")
graph.add_conditional_edges("check_question_category", routes)
graph.add_edge("coding",END)
graph.add_edge("weather",END)
graph.add_edge("Google_search",END)


In [27]:

app = graph.compile()

In [30]:
res = app.invoke({"question":"what is the temperature of bhopal"})

In [31]:
res

{'question': 'what is the temperature of bhopal',
 'category': 'weather',
 'ans': '**Current temperature in Bhopal (Madhya Pradesh, India)**  \n- **Temperature:** **≈\u202f32\u202f°C** (≈\u202f91\u202f°F)  \n- **Feels‑like (RealFeel®):** ~\u202f88\u202f°F (≈\u202f31\u202f°C)  \n\n*(Data retrieved from a recent weather query; values may fluctuate slightly as new observations are posted.)*'}